[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C21_Frontier_Pretraining_Course/01_data_curation/01_data_curation.ipynb)

# 01 · 数据清洗与去重（从零实现一条 CommonCrawl 管线）

目标：用纯 numpy 实现预训练数据管线的三大支柱——**质量过滤、MinHash/LSH 去重、n-gram 去污染**，每个都与朴素参考**对拍**。

路线：启发式过滤 → 逻辑回归质量分类器 → MinHash 估 Jaccard（核心，对拍暴力 Jaccard）→ LSH banding（S 曲线）→ n-gram 去污染 → ✏️ 练习(minhash / LSH band / n-gram 去污染 / 质量阈值 PR) → 📖 答案 → 🧪 真实 FineWeb 管线胶囊。

> 心智模型：**一篇文档 = 一个 shingle 集合；MinHash 签名 = 几个最小哈希；LSH 桶 = 一个 dict；去污染 = 在评测集 n-gram 的哈希表里查表**。我们写的是*算法与正确性*，不是亿级规模。

In [ ]:
import numpy as np
import hashlib, re
from collections import Counter, defaultdict
rng = np.random.default_rng(0)
print('环境就绪，开始构建数据管线')

## 1 · 启发式质量过滤（Gopher 风格规则）

最便宜的一层：一组硬规则判断文档「像不像值得学的文本」。我们实现长度、平均词长、符号比、重复行比例几条，组合成一个过滤器。

在干净文档 vs 各类垃圾上验证：垃圾应被拒、干净应被留。

In [ ]:
def heuristic_quality(text):
    '''返回 (是否保留, 原因)。组合若干 Gopher 风格规则；任一不过即拒。'''
    words = text.split()
    n = len(words)
    if n < 5 or n > 100_000:
        return False, f'词数 {n} 越界'
    avg_len = np.mean([len(w) for w in words])
    if avg_len < 2.0 or avg_len > 12.0:
        return False, f'平均词长 {avg_len:.1f} 异常'
    n_sym = sum(c in '#@^*<>{}[]|\\' for c in text)
    if n_sym / max(n, 1) > 0.5:
        return False, f'符号/词比 {n_sym/n:.2f} 过高'
    lines = [ln for ln in text.split(chr(10)) if ln.strip()]
    if lines:
        rep = 1 - len(set(lines)) / len(lines)
        if rep > 0.3:
            return False, f'重复行比例 {rep:.2f} 过高'
    return True, 'ok'

samples = {
    '正常文档': 'machine learning models are trained on large corpora of text data collected from many sources',
    '太短':     'click here',
    '符号刷屏': '### @@@ <<< >>> {{{ }}} [[[ ]]] ||| ^^^ *** ### @@@ <<<',
    '重复行':   chr(10).join(['buy now cheap deals'] * 10),
    '乱码':     'asdfghjklqwertyuiop zxcvbnmasdfgh qwertyuiopasdfgh',
}
for name, txt in samples.items():
    keep, why = heuristic_quality(txt)
    print(f'{name:8s} -> {"保留" if keep else "拒绝"}  ({why})')

assert heuristic_quality(samples['正常文档'])[0] is True
assert heuristic_quality(samples['太短'])[0] is False
assert heuristic_quality(samples['符号刷屏'])[0] is False
assert heuristic_quality(samples['重复行'])[0] is False
print('\n✅ 启发式过滤：正常文档留、各类明显垃圾拒')

## 2 · 分类器质量过滤：从零训逻辑回归

启发式抓不到「语法通顺但内容空洞」的垃圾。训一个**逻辑回归**分类器：词袋特征 → sigmoid → 「高质量」概率。

用合成的高质量(科技/学术词)与垃圾(营销/SEO 词)文档训练，验证它能把两类分开。

In [ ]:
# 合成两类文档：高质量(技术/学术语汇) vs 垃圾(营销/SEO 语汇)
good_vocab = 'model data training algorithm research method result analysis theory experiment'.split()
spam_vocab = 'buy cheap now click discount offer deal free guarantee limited'.split()
vocab = good_vocab + spam_vocab
vocab_idx = {w: i for i, w in enumerate(vocab)}

def make_doc(rng, vocab_pool, length=12):
    return ' '.join(rng.choice(vocab_pool, size=length))

def bow(text):
    '''词袋特征向量(在固定 vocab 上计数)。'''
    v = np.zeros(len(vocab))
    for w in text.split():
        if w in vocab_idx:
            v[vocab_idx[w]] += 1
    return v

Xg = np.array([bow(make_doc(rng, good_vocab)) for _ in range(150)])
Xs = np.array([bow(make_doc(rng, spam_vocab)) for _ in range(150)])
X = np.vstack([Xg, Xs])
y = np.concatenate([np.ones(150), np.zeros(150)])   # 1=高质量, 0=垃圾
print('特征矩阵', X.shape, '标签', y.shape)
assert X.shape == (300, len(vocab))

In [ ]:
def train_logreg(X, y, lr=0.1, epochs=300, l2=1e-3):
    '''批量梯度下降训练逻辑回归。返回 (w, b)。'''
    n, d = X.shape
    w = np.zeros(d); b = 0.0
    for _ in range(epochs):
        z = X @ w + b
        p = 1.0 / (1.0 + np.exp(-z))
        gw = X.T @ (p - y) / n + l2 * w        # 交叉熵梯度 + L2
        gb = np.mean(p - y)
        w -= lr * gw; b -= lr * gb
    return w, b

def quality_score(text, w, b):
    z = bow(text) @ w + b
    return 1.0 / (1.0 + np.exp(-z))

w, b = train_logreg(X, y)
train_acc = np.mean(((X @ w + b) > 0).astype(float) == y)
print(f'训练准确率 {train_acc:.3f}')
print(f'高质量样例得分 {quality_score(make_doc(rng, good_vocab), w, b):.3f}')
print(f'垃圾样例得分   {quality_score(make_doc(rng, spam_vocab), w, b):.3f}')
assert train_acc > 0.95, '两类可分，准确率应很高'
assert quality_score(' '.join(good_vocab), w, b) > quality_score(' '.join(spam_vocab), w, b)
print('\n✅ 逻辑回归质量分类器：高质量得分高、垃圾得分低 (FineWeb-Edu 分类器的玩具版)')

## 3 · MinHash 估 Jaccard（本模块核心）

**核心定理**：取随机哈希 h，`min_{x∈A} h(x) == min_{x∈B} h(x)` 的概率 = Jaccard(A,B)。

用 m 个独立哈希得到签名；两签名**逐位相等的比例**是 Jaccard 的**无偏估计**。我们实现它，并与暴力算的真 Jaccard **对拍**（大 m 下误差趋零）。

In [ ]:
def shingles(text, k=5):
    '''词级 k-gram 集合。'''
    toks = text.split()
    return set(' '.join(toks[i:i+k]) for i in range(max(len(toks) - k + 1, 1)))

def true_jaccard(a, b):
    return len(a & b) / len(a | b) if (a | b) else 0.0

def _hash(x, seed):
    '''确定性哈希：把 (seed, shingle) 映射到一个 64-bit 整数。'''
    h = hashlib.md5(f'{seed}:{x}'.encode()).digest()
    return int.from_bytes(h[:8], 'little')

def minhash_signature(shingle_set, m=128, seeds=None):
    '''对一个 shingle 集合算长度 m 的 MinHash 签名。'''
    if seeds is None:
        seeds = range(m)
    sig = np.empty(m, dtype=np.uint64)
    for j, s in enumerate(seeds):
        sig[j] = min(_hash(x, s) for x in shingle_set)
    return sig

def est_jaccard(sigA, sigB):
    '''签名逐位相等比例 = Jaccard 估计。'''
    return float(np.mean(sigA == sigB))

# 造一对已知相似度的文档
base = ' '.join([f'w{i}' for i in range(40)])
variant = ' '.join([f'w{i}' for i in range(8, 48)])   # 与 base 部分重叠
A, B = shingles(base), shingles(variant)
tj = true_jaccard(A, B)
for m in [16, 64, 256, 1024]:
    sigA = minhash_signature(A, m=m)
    sigB = minhash_signature(B, m=m)
    ej = est_jaccard(sigA, sigB)
    print(f'm={m:4d}: 估计 Jaccard {ej:.3f}  vs  真值 {tj:.3f}  (误差 {abs(ej-tj):.3f})')

# 大 m 下估计应逼近真值
sigA = minhash_signature(A, m=1024); sigB = minhash_signature(B, m=1024)
assert abs(est_jaccard(sigA, sigB) - tj) < 0.05, '大 m 下 MinHash 应逼近真 Jaccard'
print('\n✅ MinHash：签名相等比例无偏估计 Jaccard，m 越大越准 —— 把「比集合」变成「比短签名」')

> 误差的统计学：每一位签名是一次成功概率 = J 的伯努利试验，估计标准误差 ≈ `sqrt(J(1-J)/m)`。m=256 时误差约 0.03，已足够实用。这就是为什么真实管线常取 m=128~256。

## 4 · LSH banding：把 O(n²) 去重变可行

两两比签名仍是 O(n²)。**banding**：把 m 维签名切成 b 个 band(每个 r 行)，任一 band 完全相同 → 候选对。只比同桶的。

成为候选的概率 = `1-(1-s^r)^b`，是条 **S 曲线**。我们实现分桶，并验证候选概率曲线符合公式。

In [ ]:
def lsh_buckets(signatures, b, r):
    '''signatures: dict {doc_id: signature(np array, 长度>=b*r)}。
       返回候选对集合：任一 band 的 r 行完全相同的文档对。'''
    assert all(len(s) >= b * r for s in signatures.values())
    buckets = defaultdict(list)
    for doc_id, sig in signatures.items():
        for band in range(b):
            key = (band, tuple(int(x) for x in sig[band*r:(band+1)*r]))
            buckets[key].append(doc_id)
    candidates = set()
    for ids in buckets.values():
        for i in range(len(ids)):
            for j in range(i+1, len(ids)):
                candidates.add(tuple(sorted((ids[i], ids[j]))))
    return candidates

def candidate_prob_theory(s, b, r):
    return 1 - (1 - s**r)**b

# 经验验证 S 曲线：对给定真实相似度 s，模拟「签名逐位以概率 s 相等」，看分桶命中率
def empirical_candidate_prob(s, b, r, trials=4000, seed=0):
    g = np.random.default_rng(seed)
    hits = 0
    for _ in range(trials):
        # 一对签名：每位以概率 s 相等(等价于真 Jaccard=s 的 MinHash 行为)
        equal = g.random(b*r) < s
        # 任一 band 全部相等 -> 候选
        if any(equal[band*r:(band+1)*r].all() for band in range(b)):
            hits += 1
    return hits / trials

b, r = 20, 5     # m=100, 阈值≈(1/20)^(1/5)≈0.55
print(f'b={b}, r={r}, 阈值≈(1/b)^(1/r)={ (1/b)**(1/r):.2f}\n')
print(f"{'真实s':>6} {'理论P':>8} {'经验P':>8}")
for s in [0.3, 0.5, 0.6, 0.7, 0.9]:
    th = candidate_prob_theory(s, b, r)
    em_ = empirical_candidate_prob(s, b, r)
    print(f'{s:6.1f} {th:8.3f} {em_:8.3f}')
    assert abs(th - em_) < 0.05, '经验候选率应符合 S 曲线理论'
print('\n✅ LSH banding：候选概率符合 1-(1-s^r)^b 的 S 曲线 —— 相似的进同桶，不相似的几乎不进')

## 5 · 在小语料上跑完整去重

把 MinHash + LSH 串起来：对一个含若干**近重复**的小语料，用 LSH 找候选对、再算精确 Jaccard 确认、保留每个重复簇的一个代表。

In [ ]:
corpus = {
    0: 'the cat sat on the mat and looked at the bright morning sun',
    1: 'the cat sat on the mat and looked at the bright morning star',   # 1 的近重复(改sun->star)
    2: 'deep neural networks learn hierarchical representations from data',
    3: 'deep neural networks learn hierarchical representations from data today',  # 2 的近重复
    4: 'a completely unrelated document about cooking pasta and tomatoes',
}
sigs = {i: minhash_signature(shingles(t, k=3), m=100) for i, t in corpus.items()}
cands = lsh_buckets(sigs, b=25, r=4)
print('LSH 候选对:', sorted(cands))

# 对候选对算精确 Jaccard 确认(剔除误报)，相似度>0.5 判为重复
THRESH = 0.5
dup_pairs = []
for i, j in cands:
    tj = true_jaccard(shingles(corpus[i], k=3), shingles(corpus[j], k=3))
    if tj > THRESH:
        dup_pairs.append((i, j, round(tj, 2)))
print('确认的重复对:', dup_pairs)

# 并查集求重复簇，每簇留一个代表
parent = {i: i for i in corpus}
def find(x):
    while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
    return x
for i, j, _ in dup_pairs:
    parent[find(i)] = find(j)
keep = sorted(set(find(i) for i in corpus))
print(f'\n去重前 {len(corpus)} 篇 -> 去重后 {len(keep)} 篇 (保留代表 {keep})')
assert (0,1) in [(a,b) for a,b,_ in dup_pairs] or (0,1) in cands
assert len(keep) == 3, '应剩 3 个簇: {0,1},{2,3},{4}'
print('✅ 完整去重：LSH 缩候选 -> 精确 Jaccard 确认 -> 并查集去簇')

## 6 · n-gram 去污染：别让考题泄漏进训练集

若训练文本与评测样本共享足够长(如 13-gram)的连续片段 → 判为污染、删除。把评测集所有 n-gram 建成哈希集合，训练文档每个 n-gram 查表 O(1)。

In [ ]:
def ngrams(text, n=8):
    toks = text.split()
    return set(' '.join(toks[i:i+n]) for i in range(max(len(toks) - n + 1, 0)))

def build_contamination_index(test_samples, n=8):
    idx = set()
    for s in test_samples:
        idx |= ngrams(s, n)
    return idx

def is_contaminated(doc, contam_index, n=8):
    return len(ngrams(doc, n) & contam_index) > 0

# 评测集(考题)
test_set = [
    'what is the capital of france the answer is paris a beautiful european city',
    'compute the derivative of x squared which equals two x by the power rule',
]
contam = build_contamination_index(test_set, n=8)

train_docs = {
    'clean1':   'machine learning is a subfield of artificial intelligence research worldwide',
    'leaked':   'trivia the capital of france the answer is paris a beautiful place to visit',  # 抄了考题片段
    'clean2':   'neural networks consist of layers of interconnected computational units',
    'leaked2':  'compute the derivative of x squared which equals two x easily done',           # 抄了考题片段
}
for name, doc in train_docs.items():
    bad = is_contaminated(doc, contam, n=8)
    print(f'{name:8s} -> {"污染(删)" if bad else "干净(留)"}')

assert is_contaminated(train_docs['leaked'], contam, n=8) is True
assert is_contaminated(train_docs['leaked2'], contam, n=8) is True
assert is_contaminated(train_docs['clean1'], contam, n=8) is False
assert is_contaminated(train_docs['clean2'], contam, n=8) is False
print('\n✅ n-gram 去污染：精确捞出抄了考题片段的训练文档，不误伤干净文档')

---
## ✏️ 练习 1：从零实现 MinHash 签名

实现 `minhash_ex(shingle_set, m)`：对每个 seed(0..m-1)，取集合中所有 shingle 的 `_hash(x, seed)` 的**最小值**，组成长度 m 的签名。

必须使得两签名相等比例逼近真 Jaccard（大 m 下误差 < 0.05）。

In [ ]:
def minhash_ex(shingle_set, m=128):
    sig = np.empty(m, dtype=np.uint64)
    # TODO: for j in range(m): sig[j] = 集合中所有 x 的 _hash(x, j) 的最小值
    raise NotImplementedError
    return sig

In [ ]:
# —— 练习 1 自测 ——
A2 = shingles(' '.join([f'w{i}' for i in range(30)]), k=4)
B2 = shingles(' '.join([f'w{i}' for i in range(6, 36)]), k=4)
tj = true_jaccard(A2, B2)
sigA = minhash_ex(A2, m=1024)
sigB = minhash_ex(B2, m=1024)
ej = float(np.mean(sigA == sigB))
print(f'估计 {ej:.3f} vs 真值 {tj:.3f}')
assert sigA.shape == (1024,)
assert abs(ej - tj) < 0.05, 'MinHash 估计应逼近真 Jaccard'
print('✅ 练习 1 通过：你的 MinHash 无偏估计了 Jaccard')

## ✏️ 练习 2：调 (b, r) 命中目标阈值

给定签名长度 m 与目标阈值 t（希望 Jaccard≈t 处是 S 曲线拐点），从所有满足 `b*r==m` 的整数对里，选出**实际拐点最接近 t** 的 (b,r)。

拐点用近似公式 `(1/b)^(1/r)`。实现 `choose_br(m, t)`。

In [ ]:
def choose_br(m, t):
    '''返回使拐点 (1/b)^(1/r) 最接近 t 的 (b, r)，要求 b*r==m 且 b,r>=1。'''
    # TODO: 遍历 r 的所有约数分解 m=b*r，算拐点，返回最接近 t 的 (b,r)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
def threshold(b, r):
    return (1.0 / b) ** (1.0 / r)

b, r = choose_br(100, 0.8)
print(f'm=100, 目标阈值 0.8 -> 选 (b={b}, r={r}), 实际拐点 {threshold(b,r):.3f}')
assert b * r == 100
assert abs(threshold(b, r) - 0.8) < 0.1, '拐点应接近目标 0.8'
# 目标阈值越高，应选更大的 r(每 band 更长更难全同)
b_hi, r_hi = choose_br(100, 0.9)
b_lo, r_lo = choose_br(100, 0.4)
assert r_hi >= r_lo, '更高阈值应倾向更大的 r'
print('✅ 练习 2 通过：会调 (b,r) 把 S 曲线拐点对准目标阈值')

## ✏️ 练习 3：n-gram 去污染器

实现 `decontaminate(train_docs, test_samples, n)`：返回 `train_docs` 中**不含**任何与 `test_samples` 重叠 n-gram 的那些文档(干净的)。

复用 `ngrams` / `build_contamination_index`。

In [ ]:
def decontaminate(train_docs, test_samples, n=8):
    '''train_docs: dict {name: text}。返回干净文档的 dict。'''
    # TODO: 建污染索引；保留 ngrams(doc,n) 与索引无交集的文档
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
clean = decontaminate(train_docs, test_set, n=8)
print('保留的干净文档:', sorted(clean.keys()))
assert set(clean.keys()) == {'clean1', 'clean2'}, '应只留干净文档'
assert 'leaked' not in clean and 'leaked2' not in clean
print('✅ 练习 3 通过：去污染器删掉了所有泄题文档')

## ✏️ 练习 4：质量阈值的 Precision-Recall 权衡

用第 2 节训好的质量分类器，对一批混合(高质量+垃圾)文档按不同阈值过滤，算 **precision 与 recall**。

实现 `pr_at_threshold(scores, labels, thresh)`：返回 (precision, recall)。precision=留下里真高质量比例；recall=所有高质量中被留下比例。

In [ ]:
def pr_at_threshold(scores, labels, thresh):
    '''scores: 质量分数数组; labels: 1=高质量,0=垃圾; thresh: 保留 score>=thresh。
       返回 (precision, recall)。若没留任何文档, precision 记为 1.0。'''
    # TODO: kept = scores>=thresh; precision = (kept且label=1)/(kept); recall = (kept且label=1)/(label=1 总数)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
test_good = np.array([bow(make_doc(rng, good_vocab)) for _ in range(100)])
test_spam = np.array([bow(make_doc(rng, spam_vocab)) for _ in range(100)])
Xte = np.vstack([test_good, test_spam])
yte = np.concatenate([np.ones(100), np.zeros(100)])
scores = 1.0 / (1.0 + np.exp(-(Xte @ w + b)))

p_lo, r_lo = pr_at_threshold(scores, yte, 0.3)
p_hi, r_hi = pr_at_threshold(scores, yte, 0.7)
print(f'阈值 0.3: precision {p_lo:.3f}, recall {r_lo:.3f}')
print(f'阈值 0.7: precision {p_hi:.3f}, recall {r_hi:.3f}')
# 阈值越高 -> precision 不降(更挑) 但 recall 不升(留得少)
assert p_hi >= p_lo - 1e-9, '更高阈值 precision 应不降'
assert r_hi <= r_lo + 1e-9, '更高阈值 recall 应不升'
assert 0 <= p_lo <= 1 and 0 <= r_lo <= 1
print('✅ 练习 4 通过：看清质量过滤的 precision-recall 权衡 (预训练偏向高 precision)')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def minhash_ex(shingle_set, m=128):
    sig = np.empty(m, dtype=np.uint64)
    for j in range(m):
        sig[j] = min(_hash(x, j) for x in shingle_set)
    return sig

In [ ]:
# 练习 2 参考答案
def choose_br(m, t):
    best = None; best_gap = 1e9
    for r in range(1, m + 1):
        if m % r != 0:
            continue
        b = m // r
        gap = abs((1.0 / b) ** (1.0 / r) - t)
        if gap < best_gap:
            best_gap = gap; best = (b, r)
    return best

In [ ]:
# 练习 3 参考答案
def decontaminate(train_docs, test_samples, n=8):
    idx = build_contamination_index(test_samples, n)
    return {name: doc for name, doc in train_docs.items()
            if len(ngrams(doc, n) & idx) == 0}

In [ ]:
# 练习 4 参考答案
def pr_at_threshold(scores, labels, thresh):
    kept = scores >= thresh
    n_kept = kept.sum()
    tp = ((labels == 1) & kept).sum()
    precision = tp / n_kept if n_kept > 0 else 1.0
    recall = tp / (labels == 1).sum()
    return float(precision), float(recall)

---
## 🧪 真实数据胶囊：FineWeb 管线算一笔账

FineWeb(Penedo 2024)从 CommonCrawl 的 ~粗 90T+ token 起步，逐步过滤/去重到 **15T token** 高质量语料。用真实的「各步保留率」算一笔账，看清一条管线到底丢掉了多少、为什么去重那一步删得最狠。

In [ ]:
# FineWeb 风格各步骤的近似保留率(基于论文披露的量级，用于算账)
raw_tokens = 90e12      # 一年 CommonCrawl 抽取后的粗略量级
steps = [
    ('语言过滤(留英文)',   0.45),
    ('启发式质量过滤',     0.55),
    ('MinHash 去重',       0.30),   # 去重删得最狠：网页近重复极多
    ('细清洗/去污染等',    0.85),
]
tokens = raw_tokens
print(f'{"起始":22s} {tokens/1e12:8.1f}T token')
for name, keep in steps:
    tokens *= keep
    print(f'{name:22s} ×{keep:.2f} -> {tokens/1e12:8.2f}T token')
print(f'\n最终约 {tokens/1e12:.1f}T token (FineWeb 量级 ~15T)')

overall = tokens / raw_tokens
print(f'总保留率 {overall:.1%} —— 一条管线丢掉了约 {1-overall:.0%} 的原始 token')
assert tokens < raw_tokens * 0.2, '清洗后应大幅缩小'
# 去重是单步删除最多的(保留率最低)
keeps = dict(steps)
assert keeps['MinHash 去重'] == min(keeps.values()), '去重单步删得最狠'
print('✅ 胶囊：清洗管线丢掉绝大部分原始数据，去重是单步删除量最大的一环')

**🧪 胶囊练习**：实现 `pipeline_yield(raw, keep_rates)`：给定起始 token 数与各步保留率列表，返回 (最终 token 数, 总保留率)。

In [ ]:
def pipeline_yield(raw, keep_rates):
    # TODO: 依次乘以每个保留率；返回 (最终量, 最终/起始)
    raise NotImplementedError

In [ ]:
# 自测
final, frac = pipeline_yield(90e12, [0.45, 0.55, 0.30, 0.85])
print(f'最终 {final/1e12:.2f}T token, 总保留率 {frac:.1%}')
assert abs(frac - 0.45*0.55*0.30*0.85) < 1e-12
print('✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def pipeline_yield(raw, keep_rates):
    t = raw
    for k in keep_rates:
        t *= k
    return t, t / raw

### 小结
- **质量过滤**：启发式规则(便宜、可解释)抓明显垃圾 + 分类器(语义级)抓空洞内容；阈值在 precision/recall 间权衡，预训练偏高 precision。
- **去重**：MinHash 把「比集合」变「比短签名」(签名相等比例无偏估计 Jaccard)；LSH banding 把 O(n²) 降到接近 O(n)，(b,r) 调 S 曲线阈值。
- **去污染**：n-gram 重叠删掉与评测集重合的训练文本，防止「背题作弊」。
- **管线**：启发式→去重→分类器→去污染，顺序为省算力服务；FineWeb 用 FLOPs 对齐消融量化每步价值。

下一站：**模块 02 · Tokenizer 设计** —— 干净语料如何被切成 token，BPE 从零训练、词表大小与 fertility 的三角权衡。